In [ ]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine

os.chdir(r'C:\Users\Birchtula\PycharmProjects\PandasProject')  # 修改相对路径的位置.
# os.getcwd()

# 解决中文显示问题，下面的代码只需运行一次即可
import matplotlib as plt

plt.rcParams['font.sans-serif'] = ['SimHei']  # 如果是Mac本, 不支持SimHei的时候, 可以修改为 'Microsoft YaHei' 或者 'Arial Unicode MS'
plt.rcParams['axes.unicode_minus'] = False

# 1. Panda进阶语法 缺失值处理

## 1.1 思路1：删除缺失值

In [ ]:
# 1. 读取数据
movie_df = pd.read_csv('./data/movie.csv')
movie_df

In [ ]:
# 2. 查看数据的介绍
movie_df.columns  # 所有的列名
movie_df.info()  # 查看数据的基本信息（列名，数据类型，非缺失值数量等）
movie_df.describe()  # 查看数据的描述性统计信息（均值中位数标准差等）

In [ ]:
# 3. 删除缺失值
movie_df.dropna(inplace=True)
movie_df.dropna(axis=0)  # 删行
movie_df.dropna(axis=1)  # 删列

## 1.2 思路2：填充缺失值

In [ ]:
# 1. 查看源数据
movie_df

In [ ]:
# 2. 判断某列是否又有缺失值
pd.isnull(movie_df)  # 判断df对象 每列的每个值是否为空
pd.notnull(movie_df)  # 判断df对象 每列的每个值是否不为空

# 3. 判断某列是否是包含缺失值的列
np.all(pd.notnull(movie_df))  #整列都是True 结果是True 但凡有False，结果为False 代表该列有缺失

In [ ]:
# 4. 填充缺失值
# 写法1：填充固定值
movie_df.fillna(23).info()

In [ ]:
# 写法2：填充每列的平均值
movie_df['Revenue (Millions)'].fillna(movie_df['Revenue (Millions)'].mean(), inplace=True)
movie_df['Metascore'].fillna(movie_df['Metascore'].mean(), inplace=True)
movie_df.info()

In [ ]:
# 5. for循环的方式，使用每列的平均值来填充各列的缺失值
# 5.1 获取每个列名
for col_name in movie_df.columns:
    # 5.2 判断某列是否有缺失值
    if np.all(pd.notnull(movie_df[col_name])) == False:
        print(col_name)
        print(movie_df[col_name].mean())
        movie_df[col_name].fillna(movie_df[col_name].mean(), inplace=True)

In [ ]:
# 6. 查看结果
movie_df.info()

## 1.3 思路3：转换，然后填充或删除缺失值

In [ ]:
# 不是所有的缺失值都会用NaN表示，可能用？表示
# 思路：先转换，后删除： ? -> NaN 再删除
# 1. 加载数据
wis = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data")
wis

In [ ]:
# 2. 尝试直接删除缺失值
wis.dropna()  # ? 不是缺失值 删不掉

In [ ]:
# 3. ？-> NaN
wis.replace('?', np.nan).dropna()

# 2. 数据合并

## 2.1 思路1：contact()，能行合并和列合并

In [ ]:
# 1. 准备数据
df = pd.read_csv('./data/1960-2019全球GDP数据.csv', encoding='gbk')
df

df1 = df[:10]
df1

df2 = df[10:20]
df2

In [ ]:
# 2. 通过concat()合并数据
# 列合并默认参考列名
new_df = pd.concat([df1, df2], axis=0)  # axis=0,列合并
new_df = pd.concat([df1, df2], )  # 效果同上
# 行合并默认参考索引
new_df = pd.concat([df1, df2], axis=1)  #行合并
new_df

In [ ]:
# 3. 修改df2的列索引
df2.index = [0, 1, 2, 3, 4, 11, 12, 13, 14, 15]
df2

In [ ]:
# 4. 再次进行 行合并
pd.concat([df1, df2], axis=1)

In [ ]:
# 5. 再次进行行合并
pd.concat([df1, df2], axis=1, join='outer')  # 默认是：满外连接，左表全集+右表全集+交集
# 内连接：只要交集
pd.concat([df1, df2], axis=1, join='inner')

## 2.2 思路2：merge() 只能进行 行合并

In [ ]:
# 1. 准备数据集
df1 = pd.DataFrame({
    'key1': ['K0', 'K0', 'K1', 'K2'],
    'key2': ['K0', 'K1', 'K0', 'K1'],
    'A': ['A0', 'A1', 'A2', 'A3'],
    'B': ['B0', 'B1', 'B2', 'B3']
})
df2 = pd.DataFrame({
    'key1': ['K0', 'K1', 'K1', 'K2'],
    'key2': ['K0', 'K0', 'K0', 'K0'],
    'C': ['C0', 'C1', 'C2', 'C3'],
    'D': ['D0', 'D1', 'D2', 'D3']
})
# 2. 查看数据
df1
df2

In [ ]:
# 3. merge()函数的默认合并方式
pd.merge(df1, df2, how='inner', on=['key1', 'key2'])
pd.merge(df1, df2)  # 效果同上 默认inner join 且参考同名列

In [ ]:
# 4. 内连接，指定合并字段
pd.merge(df1, df2, how='inner', on='key2')  # 内连接，关键字段为key2

In [ ]:
# 5. 外连接，指定合并字段
pd.merge(df1, df2, how='outer', on=['key1', 'key2'])  # 满外连接 关键字段key1 key2

# 左外连接 = 左表全集 + 交集
# pd.merge(df1,df2,how='left',on=['key1','key2'])

In [60]:
# 6. merge()函数的其他写法
df1.merge(df2,how='inner',on='key1')    #效果同上，即df1.merge(df2)

# concat() pd.concat([df1,df2...])  可以同时拼接多个，可以行合并和列合并，默认是外连接
pd.concat([df1,df2,df1])

,key1,key2,A,B,C,D
0,K0,K0,A0,B0,NaN,NaN
1,K0,K1,A1,B1,NaN,NaN
2,K1,K0,A2,B2,NaN,NaN
3,K2,K1,A3,B3,NaN,NaN
0,K0,K0,NaN,NaN,C0,D0
1,K1,K0,NaN,NaN,C1,D1
2,K1,K0,NaN,NaN,C2,D2
3,K2,K0,NaN,NaN,C3,D3
0,K0,K0,A0,B0,NaN,NaN
1,K0,K1,A1,B1,NaN,NaN
